# 🏁 24 · Capstone: one dataset, three methods — a complete 02-620-style project
### From a biological question to an honest report: an evaluation contract, nested cross-validation, uncertainty, stability, a sealed test set, a network-learning alternative, a report template, a peer-review rubric and the whole course on one page

**BiologyToML** · Part E — Bayesian learning, neural networks and model selection · Module 24 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/24_capstone_project.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn |
| **Prerequisites** | All modules; at least 00–14 and 23 (model selection). Section 9 uses Modules 15–18. |
| **Estimated time** | 6–10 h as a project (≈ 4–5 h to run and read the worked examples and the course summary; the rest is your own project and report) |
| **02-620 lectures covered** | `001_intro` slides 9–10 (the course project) and every deck — see the self-assessment in Section 11 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Real data only: the Golub et al. (1999) leukaemia expression data (72 patients × 3,571 genes, `load_golub`) for the
> worked project, and the Sachs et al. (2005) single-cell signalling data (`load_sachs`) for the alternative mini-project.
> Offline, both loaders fall back to clearly labelled SIMULATED stand-ins — check the printed `source` lines.
> The 34 patients of Golub's independent test set are **sealed** until Section 8: please do not look at them earlier.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Plan** a machine-learning project the way 02-620 asks: a question, one data set, three methods, a proposal, a report and peer review.
2. **Write** an evaluation contract before touching the methods: target, unit of replication, split, metric, baseline and what would count as success.
3. **Compare** three methods from different families (L1-logistic regression, random forest, RBF support vector machine) with nested cross-validation, identical folds and a fixed tuning budget.
4. **Quantify** uncertainty honestly: fold-to-fold spread, a corrected paired comparison, confidence intervals, a bootstrap of the test set, McNemar's test and calibration.
5. **Interpret** the selected genes and their stability across folds, and **state** what the models can and cannot claim biologically.
6. **Learn** a signalling network from real single-cell data with three methods (Chow–Liu, BIC hill climbing, graphical lasso) and **score** them against a consensus network.
7. **Write and review** a project report with a template and a rubric, and **check** your understanding of every 02-620 lecture with a self-assessment that links each deck to its modules.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · The 02-620 project | what is asked, what a good project looks like | 15 min |
| 2 · Question and evaluation contract | Golub ALL vs AML; development set and sealed test set | 15 min |
| 3 · Exploratory analysis | PCA and a clustered heatmap of the development set | 20 min |
| 4 · Three methods, one fair protocol | pipelines, tuning grids, nested CV from scratch | 30 min |
| 5 · Nested-CV results | accuracies, chosen settings | 15 min |
| 6 · Comparison with honest uncertainty | paired differences, corrected t-test, calibration | 30 min |
| 7 · Interpretation and stability | which genes, how stable, what can be claimed | 25 min |
| 8 · Unsealing the test set | final models, CIs, bootstrap, McNemar | 20 min |
| 9 · Alternative mini-project: a signalling network | Sachs data: Chow–Liu, hill climbing, graphical lasso | 45 min |
| 10 · Writing it up | results table, report template, peer-review rubric, presentation checklist | 15 min |
| 11 · The course in one page | method map, three ideas, self-assessment for every deck | 40 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, project ideas | 10 min |
| Your own project | apply the template to a data set of your choice, write the report, review a peer's | 75–315 min |

## 🧫 The biological question

In 1999, acute leukaemias were diagnosed by experts combining a cell's appearance under the microscope with
histochemistry and immunophenotyping. Telling **acute lymphoblastic leukaemia (ALL)** from **acute myeloid leukaemia
(AML)** matters because the two diseases are treated differently, and no single test was decisive. Golub *et al.* asked
whether the **expression of thousands of genes**, measured at once on a microarray, could make that call automatically
("class prediction") — and whether the classes could have been found without knowing them ("class discovery"). Their
paper became one of the founding data sets of machine learning in biology; you have met it in many modules of this course.

This capstone treats it as a **complete research project**, the way the 02-620 team project asks: one data set, three
methods, a fair and honest comparison, an interpretation that does not over-claim, and a written report reviewed by
peers. The emphasis is not on squeezing out the last percent of accuracy — with 72 patients that would be meaningless —
but on **every decision that makes a result trustworthy**: what is the unit of replication, which data may be used for
which choice, how uncertain is each number, and what could a biologist actually do with the answer?

For students who prefer networks to classifiers, Section 9 is a second, complete mini-project on the Sachs *et al.*
(2005) single-cell signalling data: learn the network with three methods and score them against what is known.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.colors import ListedColormap
from scipy import stats
from scipy.cluster import hierarchy
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.covariance import GraphicalLasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They load the two real data sets (with offline fallbacks), compute accuracy, confusion counts, precision and recall, and draw small graphs with Matplotlib. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def simulate_sachs_like(rng: np.random.Generator, n: int = 2603, kind: str = "continuous") -> pd.DataFrame:
    """Simulated stand-in for the Sachs data: a linear-Gaussian model on the consensus network."""
    order = ["pip3", "plc", "pip2", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]  # topological
    parents = {p: [a for a, b in SACHS_CONSENSUS_EDGES if b == p] for p in SACHS_PROTEINS}
    data = {}
    for node in order:
        value = rng.normal(0, 1, n)
        for par in parents[node]:
            value = value + rng.choice([-1, 1]) * rng.uniform(0.4, 0.8) * data[par]
        data[node] = (value - value.mean()) / value.std()
    df = pd.DataFrame({p: data[p] for p in SACHS_PROTEINS})
    if kind == "discrete":
        df = df.apply(lambda c: pd.qcut(c, 3, labels=False)).astype(int)
    return df


def load_sachs(kind: str = "continuous", observational_only: bool = True) -> dict:
    """Sachs et al. (2005) single-cell phospho-protein measurements (flow cytometry).

    ``kind='continuous'`` gives log-scale intensities (7,466 cells), ``'discrete'`` the
    low/medium/high (0/1/2) version (5,400 cells).  With ``observational_only=True`` only
    cells without a targeted perturbation are returned.  Returns a dict with ``data``
    (DataFrame, 11 proteins), ``intervention``, ``edges`` (consensus network) and ``source``.
    """
    path = fetch_course_file(f"sachs_cytometry_{kind}.csv.gz")
    if path is None:
        df = simulate_sachs_like(np.random.default_rng(2005), kind=kind)
        ivn, source = np.array(["none"] * len(df)), "SIMULATED stand-in (offline)"
    else:
        full = pd.read_csv(path)
        df, ivn, source = full[SACHS_PROTEINS], full["intervention"].to_numpy(), "Sachs et al. 2005 (real data)"
    if observational_only:
        keep = ivn == "none"
        df, ivn = df[keep].reset_index(drop=True), ivn[keep]
    return dict(data=df, intervention=ivn, edges=list(SACHS_CONSENSUS_EDGES), source=source)


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'simulate_sachs_like', 'load_sachs', 'standardize', 'accuracy', 'confusion_counts', 'precision_recall_f1', 'draw_graph']))

---
## 1 · The 02-620 project: what is asked, and what a good project looks like

> 📍 **Where we are.** You have worked through every lecture of 02-620 (Modules 00–23). The course ends with a team
> project; this notebook walks through one complete example and gives you the templates to run your own.

The first lecture describes the project in two slides (`001_intro` slides 9–10, with speaker notes): *"In the course
project, you will analyze a dataset with three methods of your choice as a team"* of 2–4 students. It is worth 25 % of
the grade and has five deliverables:

| Deliverable | What it must contain | Where this notebook helps |
|---|---|---|
| **Proposal** | the question, the data, the three methods, how you will evaluate them | Sections 2 and 4 (the contract and the protocol) |
| **Peer review of proposals** | a constructive critique of another team's plan | the rubric in Section 10 |
| **Presentation** (final week) | the story: question → data → methods → results → limits | the checklist in Section 10 |
| **Final report** | methods, results with uncertainty, interpretation, limitations | the template in Section 10, Sections 5–8 |
| **Peer review of reports** | a structured assessment of another team's report | the rubric in Section 10 |

**What a good project looks like.** Experience with student projects (and with papers) suggests five marks of quality:

1. **A real question, not a method demo.** "Can expression distinguish ALL from AML, and which genes carry the signal?"
   rather than "we tried three classifiers".
2. **Methods from different families, chosen for a reason** — e.g. a sparse linear model (interpretable), an ensemble
   of trees (non-linear, interactions), a kernel method (flexible boundary) — or for networks: a tree, a general DAG
   and an undirected model.
3. **An evaluation decided in advance** and applied identically to every method (same folds, same budget, no test-set
   peeking).
4. **Uncertainty on every number** and a comparison against a trivial baseline.
5. **Calibrated claims**: say what the analysis shows, what it does not show, and what experiment would settle it.

### 📑 Slide connection — `001_intro`, slides 9–10

- **What the slides show.** Grading: homework 50 % (written + Python), two non-cumulative midterms (12.5 % each), project 25 %, attendance surveys +2 %. The project: analyse a data set with three methods of your choice as a team; proposal, peer review of the proposal, presentation in the final week, final report, peer review of the final report; more information on suggested projects and formats to follow. The speaker notes add that teams have 2–4 students and that all programming is in Python.
- **What this notebook adds.** A complete worked project on a real data set, with every step the slides list (proposal → review → presentation → report → review) turned into a concrete artefact: an evaluation contract, a nested-CV protocol, an uncertainty analysis, a report template and a review rubric.
- **Exam tip.** The project is graded on the *quality of reasoning* as much as on results: a well-argued negative result ("the three methods are indistinguishable on 72 patients") is a good result if the uncertainty analysis supports it.

---
## 2 · Step 1 — the question and an evaluation contract

> 📍 **Where we are.** Before any model is fitted, a good project writes down *how it will be judged*. [Module 23](23_model_selection.ipynb) showed
> how easily performance estimates become optimistic when this is decided afterwards (selection bias, test-set reuse).

An **evaluation contract** fixes: the **question** and the **target**; the **unit of replication** (what counts as one
independent observation — here a patient); the **split** (which data are used for which decision); the **metric**; the
**baseline** every method must beat; and what will count as **success**. Golub *et al.* already split their 72 patients:
38 bone-marrow samples to build the predictor and an **independent set of 34**, collected separately and including
peripheral-blood samples. We adopt that split. The 34 independent patients are our **sealed test set**: they will be
used **once**, at the end (Section 8), exactly as the 02-620 model-selection slides demand (*"You should not use test
data during training with CV. Otherwise, you would be overfitting!"*, `015_model-selection` slide 33).

In [ ]:
# ── 2.1 Load the data, split it and write the contract ──
golub = load_golub()
print("Source:", golub["source"])
X_all, y_all, genes = golub["X"], golub["y"], golub["genes"]
dev = golub["is_train"]                                   # Golub's 38 training patients = our development set
X_dev, y_dev = X_all[dev], y_all[dev]
X_test, y_test = X_all[~dev], y_all[~dev]                 # 34 independent patients — SEALED until Section 8
CONTRACT = {
    "Question": "Does a sample's expression profile predict ALL vs AML, and which genes carry the signal?",
    "Unit of replication": "one patient (one sample per patient)",
    "Target": "y = 1 for AML (the 'positive' class), y = 0 for ALL",
    "Development set": f"{len(y_dev)} patients ({np.sum(y_dev == 0)} ALL / {np.sum(y_dev == 1)} AML), {X_dev.shape[1]:,} genes",
    "Sealed test set": f"{len(y_test)} patients ({np.sum(y_test == 0)} ALL / {np.sum(y_test == 1)} AML) — used once, in Section 8",
    "Primary metric": "accuracy (fraction of patients classified correctly) with a 95 % confidence interval",
    "Secondary metrics": "AML precision and recall, confusion matrix, Brier score for probabilistic outputs",
    "Baseline": f"always predict the majority class ALL (development accuracy {np.mean(y_dev == 0):.3f})",
    "Model selection": "nested CV on the development set only: outer 5-fold × 2 repeats, inner 3-fold",
    "Fairness": "same folds for every method; 4 hyper-parameter settings each; genes chosen inside each training fold",
    "Success": "beat the baseline by more than the uncertainty; claim a winner only if the paired comparison supports it",
}
for key, value in CONTRACT.items():
    print(f"{key:>20s} │ {value}")

🔍 **What just happened?** We wrote the rules *before* seeing any result. Three of them deserve a comment.

* **Unit of replication.** Each patient contributes one sample, so patients are independent and a random split of
  samples is a split of patients. If some patients had several samples (e.g. bone marrow *and* blood), all samples of
  a patient would have to stay in the same fold — otherwise the test fold contains near-copies of training samples
  (Module 23: *grouped* data matter more than the number of folds).
* **Baseline.** 27 of the 38 development patients have ALL, so a "classifier" that always answers ALL is right 71 % of
  the time. Accuracy must be read against that number, not against 50 %.
* **Two levels of evaluation.** Nested CV on the development set *estimates* performance and chooses
  hyper-parameters; the sealed test set *checks* that estimate once on patients collected separately.

✏️ **Your turn.** For your own project, fill in the same dictionary *before* you write any modelling code, and paste it
into your proposal. Which entry was hardest to decide?

---
## 3 · Exploratory analysis of the development set

> 📍 **Where we are.** Before supervised learning, look at the data without labels (Part C): is there structure, are
> there outliers, does anything look technical? We use **only the 38 development patients** — even unsupervised
> exploration of the test set could influence later choices.

Two tools from Part C: **PCA** ([Module 14](14_pca_dimensionality_reduction.ipynb)) for a global picture, and
**hierarchical clustering** ([Module 12](12_clustering_kmeans_hierarchical.ipynb); average
linkage, correlation distance) of the 50 most variable genes for a heatmap. Labels are used only to colour the plots
afterwards.

In [ ]:
# ── 3.1 PCA of the development set (labels only for colouring) ──
Xd_c = X_dev - X_dev.mean(axis=0)
V_d, s_d, Wt_d = np.linalg.svd(Xd_c, full_matrices=False)          # PCA by SVD (Module 14)
pc_d = V_d[:, :2] * s_d[:2]
frac_d = s_d ** 2 / np.sum(s_d ** 2)
print(f"PC1 explains {frac_d[0]:.1%}, PC2 {frac_d[1]:.1%} of the variance among the 38 development patients")
sep = [(pc_d[y_dev == 1, k].mean() - pc_d[y_dev == 0, k].mean()) / pc_d[:, k].std() for k in range(2)]
print(f"standardised AML–ALL difference along PC1: {sep[0]:+.2f}, along PC2: {sep[1]:+.2f} (sign is arbitrary)")

fig, ax = plt.subplots(figsize=(5, 3.6))
for c, lab in enumerate(["ALL", "AML"]):
    m = y_dev == c
    ax.scatter(pc_d[m, 0], pc_d[m, 1], s=24, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=lab)
ax.set_xlabel(f"PC1 ({frac_d[0]:.0%} of variance)"); ax.set_ylabel(f"PC2 ({frac_d[1]:.0%})")
ax.set_title("Development set (38 patients), coloured afterwards"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 Clustered heatmap of the 50 most variable genes (development set) ──
top_var = np.argsort(X_dev.var(axis=0))[::-1][:50]
Z_top = standardize(X_dev[:, top_var])[0]                  # z-score each gene for display
link_pat = hierarchy.linkage(Z_top, method="average", metric="correlation")
link_gen = hierarchy.linkage(Z_top.T, method="average", metric="correlation")
o_pat, o_gen = hierarchy.leaves_list(link_pat), hierarchy.leaves_list(link_gen)
clusters = hierarchy.fcluster(link_pat, 2, criterion="maxclust")
print("two-cluster cut of the patient dendrogram vs diagnosis:")
print(pd.crosstab(pd.Series(clusters, name="cluster"), pd.Series(np.where(y_dev == 1, "AML", "ALL"), name="diagnosis")).to_string())

fig = plt.figure(figsize=(10, 4.2))
ax_d = fig.add_axes([0.08, 0.80, 0.80, 0.17])
hierarchy.dendrogram(link_pat, ax=ax_d, color_threshold=0, above_threshold_color="0.3", no_labels=True)
ax_d.axis("off"); ax_d.set_title("Average-linkage clustering of patients (correlation distance, 50 most variable genes)")
ax_c = fig.add_axes([0.08, 0.75, 0.80, 0.04])
ax_c.imshow(y_dev[o_pat][None, :], aspect="auto", cmap=ListedColormap(CLASS_COLORS[:2]))
ax_c.set_xticks([]); ax_c.set_yticks([0], ["AML"], fontsize=7)
ax_h = fig.add_axes([0.08, 0.08, 0.80, 0.65])
im = ax_h.imshow(Z_top[o_pat][:, o_gen].T, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax_h.set_xlabel("patients (dendrogram order)"); ax_h.set_ylabel("50 genes (clustered)"); ax_h.set_yticks([])
fig.colorbar(im, cax=fig.add_axes([0.90, 0.08, 0.015, 0.65]), label="z-score")
plt.show()

🔍 **What just happened?** Without labels, the development patients are far from a featureless cloud: in the PCA plot
the AML patients (red) gather on one side of PC1, and the clustered heatmap shows blocks of genes that are high in one
group of patients and low in the others. The two-cluster cut of the dendrogram (table) puts all 11 AML patients into one
cluster, together with 9 ALL patients, while the other cluster is pure ALL. So the diagnosis is a major — but not the
only — source of variation among the most variable genes (ALL itself comprises B- and T-lineage subtypes, which Golub
*et al.* also discovered without labels). For the supervised task this is encouraging: the classes differ strongly
in at least some genes. Equally important is what the plots do *not* show: no patient sits far from all the others
(no obvious outlier or failed array).

⚠️ This exploration is **hypothesis-generating**. It used the most *variable* genes, chosen without labels, so it does
not bias the supervised evaluation; choosing genes by their ALL/AML difference here and then cross-validating would
(Module 23, Ambroise & McLachlan 2002).

### 📑 Slide connection — `011_PCA_mz_031825 & 006_clustering`, slides 011: 2–5, 39–40; 006: 35–52

- **What the slides show.** PCA reduces $K$ features to a few PCs for visualisation (011 slides 2–5) by projecting the centred data onto the top eigenvectors (slides 39–40). Hierarchical agglomerative clustering merges the closest clusters step by step and draws a dendrogram; single, complete and average linkage define cluster distances (006 slides 35–52).
- **What this notebook adds.** Both tools used as the *first step of a project*: on the development set only, with labels added afterwards, and with the caveat that exploration must not leak label information into the later evaluation.
- **Exam tip.** In a report, an exploratory figure earns its place if it changes or supports a decision (is the problem feasible? are there outliers or batches?). Say which samples and which genes it used.

---
## 4 · Three methods, one fair protocol

> 📍 **Where we are.** The contract is fixed and the data look sensible. Now we choose three methods and a protocol
> that treats them identically.

**The three methods** come from three different families of the course:

| Method | Module | Family (course map) | Model and estimation | Hyper-parameters tuned here (4 settings each) |
|---|---|---|---|---|
| **L1-regularised logistic regression** | [07](07_ridge_lasso_sparse_regression.ipynb), [09](09_logistic_regression.ipynb) | probabilistic, discriminative: models $P(Y\mid X)$ | Bernoulli likelihood + Laplace prior → MAP; selects genes (sparse $w$) | inverse penalty $C\in\{0.1, 0.3, 1, 3\}$ |
| **Random forest** | [10](10_decision_trees_random_forests.ipynb) | non-probabilistic ensemble (vote of trees) | greedy trees on bootstrap samples with random feature subsets | features per split $\in\{\sqrt{k}, 0.2k\}$ × minimum leaf size $\in\{1, 3\}$ |
| **RBF support vector machine** | [11](11_support_vector_machines.ipynb) | non-probabilistic, max-margin (cousin of logistic regression) | soft-margin QP in the dual, Gaussian kernel | $C\in\{1, 10\}$ × kernel width $\gamma\in\{0.2/k, 1/k\}$ |

**Shared preprocessing, inside every training fold.** With 3,571 genes and a few dozen patients, every method first
keeps the $k=200$ genes with the largest two-sample $t$-statistic between ALL and AML,

$$t_j = \frac{\bar x_{j,\mathrm{AML}} - \bar x_{j,\mathrm{ALL}}}{\sqrt{s^2_{j,\mathrm{AML}}/n_{\mathrm{AML}} + s^2_{j,\mathrm{ALL}}/n_{\mathrm{ALL}}}},$$

*In words:* the difference of the class means of gene $j$ divided by its standard error — how many "noise units" apart
the two groups are. The crucial word is **inside**: the $t$-statistics are recomputed from the training part of each
fold only, so the held-out patients never influence which genes are used (Module 23's selection-bias lesson). We
build each method as a scikit-learn *pipeline* (filter → scale → classifier) so that the whole chain is refitted in
every fold.

⚠️ The 02-620 model-selection slide lists *"SVM: NA"* for model selection (`015` slide 3). In practice an SVM has
important choices — $C$, the kernel and its parameters, class weights, preprocessing — and here it gets the same
tuning budget as the other methods.

In [ ]:
# ── 4.1 The in-fold gene filter and a version-proof L1-logistic regression ──
class TopGenesByT(BaseEstimator, TransformerMixin):
    """Keep the k genes with the largest |t|-statistic between the two classes of the data passed to fit()."""

    def __init__(self, k=200):
        self.k = k

    def fit(self, X, y):
        a, b = X[y == 0], X[y == 1]
        se = np.sqrt(a.var(axis=0, ddof=1) / len(a) + b.var(axis=0, ddof=1) / len(b))
        t = np.divide(b.mean(axis=0) - a.mean(axis=0), se, out=np.zeros(X.shape[1]), where=se > 0)
        self.keep_ = np.sort(np.argsort(-np.abs(t))[:self.k])      # constant genes get t = 0
        return self

    def transform(self, X):
        return X[:, self.keep_]


def l1_logistic(C):
    """L1-penalised (lasso) logistic regression that works on old and new scikit-learn versions."""
    version = tuple(int(v) for v in re.findall(r"\d+", sklearn.__version__)[:2])
    if version >= (1, 8):                     # 'penalty' is deprecated from 1.8 on; l1_ratio = 1 means pure L1
        return LogisticRegression(C=C, l1_ratio=1.0, solver="liblinear", random_state=SEED)
    return LogisticRegression(C=C, penalty="l1", solver="liblinear", random_state=SEED)


filt = TopGenesByT(200).fit(X_dev, y_dev)
print("genes kept when the filter sees all 38 development patients:", len(filt.keep_),
      "| e.g.", ", ".join(genes[filt.keep_[:4]]))

In [ ]:
# ── 4.2 The candidate settings: a fixed budget of 4 per method ──
K_GENES = 200
CANDIDATES = {
    "L1-logistic": [(f"C={C}", make_pipeline(TopGenesByT(K_GENES), StandardScaler(), l1_logistic(C)))
                    for C in (0.1, 0.3, 1.0, 3.0)],
    "Random forest": [(f"max_features={mf}, min_leaf={leaf}",
                       make_pipeline(TopGenesByT(K_GENES),
                                     RandomForestClassifier(n_estimators=100, max_features=mf,
                                                            min_samples_leaf=leaf, random_state=SEED)))
                      for mf in ("sqrt", 0.2) for leaf in (1, 3)],
    "RBF SVM": [(f"C={C}, gamma={g:g}/k", make_pipeline(TopGenesByT(K_GENES), StandardScaler(),
                                                        SVC(C=C, kernel="rbf", gamma=g / K_GENES)))
                for C in (1, 10) for g in (0.2, 1.0)],
}
for name, cands in CANDIDATES.items():
    print(f"{name:>14s}: " + " | ".join(label for label, _ in cands))

### 4.3 Nested cross-validation, written out

🧮 **Math refresher — nested CV ([Module 23](23_model_selection.ipynb)).** Cross-validation splits the data into $F$ folds; each fold is held out
once while the model is fitted on the rest (slide 34 of `015`). If the *same* CV scores are used both to **choose**
hyper-parameters and to **report** performance, the report is optimistic: we picked the setting that happened to fit
those folds best. **Nested** CV adds an inner loop:

1. **Outer loop** (5 folds, repeated twice with different shuffles): hold out one outer fold as a mini test set.
2. **Inner loop** (3 folds) *on the outer-training part only*: score each of the 4 settings; pick the best.
3. Refit the chosen setting on the whole outer-training part and predict the held-out outer fold.

The outer-fold accuracies estimate how well *the whole procedure, tuning included,* works on new patients. Every method
sees **exactly the same outer and inner folds**, so their results can be compared fold by fold (a *paired* design).

In [ ]:
# ── 4.3 Nested cross-validation from scratch (same folds for every method) ──
def nested_cv(candidates, X, y, n_outer=5, n_repeats=2, n_inner=3, seed=SEED):
    """Nested CV: inner CV picks one of each method's settings; the outer fold scores the refitted choice.

    Returns {method: dict(acc, choice, proba, models, test_idx)} with one entry per outer fold (proba holds
    out-of-fold P(AML) per repeat for methods that output probabilities, NaN otherwise).
    """
    res = {name: dict(acc=[], choice=[], proba=np.full((n_repeats, len(y)), np.nan), models=[], test_idx=[])
           for name in candidates}
    for r in range(n_repeats):
        outer = StratifiedKFold(n_outer, shuffle=True, random_state=seed + r)
        for tr, te in outer.split(X, y):
            inner = list(StratifiedKFold(n_inner, shuffle=True, random_state=seed + 100 + r).split(X[tr], y[tr]))
            for name, cands in candidates.items():
                inner_acc = [np.mean([np.mean(clone(model).fit(X[tr][a], y[tr][a]).predict(X[tr][b]) == y[tr][b])
                                      for a, b in inner]) for _, model in cands]
                best = int(np.argmax(inner_acc))                  # ties → the first (simplest-listed) setting
                fitted = clone(cands[best][1]).fit(X[tr], y[tr])
                res[name]["acc"].append(accuracy(y[te], fitted.predict(X[te])))
                res[name]["choice"].append(best)
                if hasattr(fitted, "predict_proba"):
                    res[name]["proba"][r, te] = fitted.predict_proba(X[te])[:, 1]
                res[name]["models"].append(fitted)
                res[name]["test_idx"].append(te)
    return res


t0 = time.perf_counter()
NCV = nested_cv(CANDIDATES, X_dev, y_dev)
print(f"nested CV: {len(NCV['L1-logistic']['acc'])} outer folds × 3 methods in {time.perf_counter() - t0:.0f} s")

### 📑 Slide connection — `015_model-selection_mz_040725 & 015_model-selection--extra`, slides 015: 2–3, 30–37; extra: 3–6

- **What the slides show.** Learning = parameter estimation given a model; model selection = choosing the model (slide 2), with a knob per method — features, K, tree size, 'SVM: NA', graph structure, number of clusters/states, architecture (slide 3). CV: split the *training* data into training and validation sets, score each model on the validation set, pick the best, and never use test data during CV (slides 31–33); k-fold CV and LOOCV (slides 34–36); larger k = more accurate assessment but k trainings; k = 5 or 10 common; LOOCV for small data (slide 37). The extra slides' case study: Golub's microarray data (7,130 genes, 72 samples: 47 ALL, 25 AML) with three classifiers — kNN, a Gaussian classifier and logistic regression.
- **What this notebook adds.** A from-scratch nested CV that keeps tuning inside the outer training folds, identical folds for all methods, a fixed tuning budget, and in-fold gene selection — on the Dudoit-preprocessed version of the same Golub data (3,571 of the original 7,129 probes pass the standard filters). The correction to 'SVM: NA' is applied: the SVM is tuned.
- **Exam tip.** Nested CV estimates the performance of a *procedure* (including its tuning), not of one fitted model. More folds change the cost and the bias/variance of the *estimate*; whether folds respect the unit of replication matters more.

### 📑 Slide connection — `002_regression & 004_classification-LR`, slides 002: 61–69; 004: 22–31

- **What the slides show.** Regularised regression: arg min $(y-Xw)^\top(y-Xw)+\lambda F(w)$; sparsity because few SNPs should matter; the lasso (L1) shrinks *and* selects (002 slides 61–69). Logistic regression is trained by maximising the conditional likelihood with gradient ascent (concave); MAP with prior $W\sim N(0,\sigma^2I)$ adds an L2 term with $\lambda=1/(2\sigma^2)$ that reduces overfitting when data are sparse (004 slides 22–31).
- **What this notebook adds.** The L1 penalty applied to *logistic* regression (scikit-learn's $C$ is an inverse penalty, $C\propto1/\lambda$), tuned by inner CV and used to select genes; Section 7 checks how stable that selection is.
- **Exam tip.** Small $C$ = strong penalty = few genes (possibly none: then the model predicts the majority class). A selected gene is a useful predictor *given the others*, not a proven driver.

### 📑 Slide connection — `005_decision-tree & 007_SVM`, slides 005: 56–58; 007: 32–41, 58–67

- **What the slides show.** Decision trees are high-variance; a random forest grows many trees on bootstrap samples with random feature subsets and takes a majority vote; on simulated data 500 shallow trees perform well (005 slides 56–58). SVMs: soft margin with slack variables and the constant $C$ for non-separable data (007 slides 32–41); kernels map the data to a feature space where they become separable, and margin maximisation acts as regularisation (slides 58–67).
- **What this notebook adds.** Both methods inside the same pipeline and protocol as the probabilistic model, with honest tuning of their hyper-parameters (features per split and leaf size; $C$ and $\gamma$).
- **Exam tip.** Random-forest 'probabilities' are vote fractions and SVM outputs are margins, not probabilities — both methods are non-probabilistic in the course's sense; check calibration before reading their scores as probabilities.

---
## 5 · Nested-CV results

In [ ]:
# ── 5.1 Outer-fold accuracies and the settings chosen by the inner loop ──
base_acc = np.array([np.mean(y_dev[te] == 0) for te in NCV["L1-logistic"]["test_idx"]])   # majority-class baseline
rows = [("Baseline (always ALL)", base_acc)] + [(name, np.array(r["acc"])) for name, r in NCV.items()]
print(f"{'method':>22s}   mean acc   SD over folds   min    folds with 100 %")
for name, a in rows:
    print(f"{name:>22s}   {a.mean():8.3f}   {a.std(ddof=1):13.3f}   {a.min():.3f}   {np.sum(a == 1):2d} / {len(a)}")
print("\nhow often each setting was chosen by the inner CV:")
for name, cands in CANDIDATES.items():
    counts = np.bincount(NCV[name]["choice"], minlength=len(cands))
    print(f"{name:>14s}: " + " | ".join(f"{lab}: {c}" for (lab, _), c in zip(cands, counts)))

fig, ax = plt.subplots(figsize=(7, 3.4))
names = [n for n, _ in rows]
for f in range(len(base_acc)):                             # one grey line per outer fold (paired design)
    ax.plot(range(len(rows)), [a[f] for _, a in rows], color="0.8", lw=0.8, zorder=1)
for i, (name, a) in enumerate(rows):
    ax.scatter(np.full(len(a), i) + np.linspace(-0.12, 0.12, len(a)), a, s=16, zorder=2,
               color=["0.5", "tab:blue", "tab:green", "tab:orange"][i])
    ax.errorbar(i + 0.25, a.mean(), yerr=a.std(ddof=1), fmt="D", color="k", ms=5, capsize=3, zorder=3)
ax.set_xticks(range(len(rows)), names, fontsize=8); ax.set_ylabel("accuracy on the outer fold")
ax.set_ylim(0.4, 1.04); ax.set_title("Nested CV on the 38 development patients (10 outer folds; ◆ mean ± SD)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All three methods are far above the majority-class baseline in every summary (compare the
printed means with the baseline's). Many outer folds are classified perfectly — with only 7–8 patients per outer fold,
one mistake costs 12–14 percentage points, which is why the per-fold accuracies jump in big steps and the fold-to-fold
standard deviation is large. The inner loop did not always pick the same setting (the counts above): with ~25 training
patients per inner split, several settings tie or nearly tie, and the "best" one varies from fold to fold. That is
normal and is exactly why tuning must be repeated inside each outer fold.

Which method is "best"? The RBF SVM classified every outer fold perfectly; the other two average a little above 92 %,
up to about 8 percentage points lower. Before ranking them, we need to know how large a difference could arise by
chance on 38 patients.

---
## 6 · Comparison with honest uncertainty

> 📍 **Where we are.** Section 5 gave one number per method and fold. Here we ask whether the differences between
> methods are larger than the noise, and whether the predicted probabilities can be trusted.

**Paired differences.** Because every method saw the same folds, we compare them fold by fold: $d_f = \mathrm{acc}_A(f)
- \mathrm{acc}_B(f)$. A naive $t$-test would treat the $F$ differences as independent, but they are not: the training
sets of different folds overlap heavily, so the differences are positively correlated and the naive standard error is
too small. Nadeau & Bengio (2003) proposed a simple correction:

$$\mathrm{SE}_{\mathrm{corrected}} = \sqrt{\left(\frac{1}{F} + \frac{n_{\mathrm{test}}}{n_{\mathrm{train}}}\right) s_d^2}
\qquad\text{instead of}\qquad \mathrm{SE}_{\mathrm{naive}} = \sqrt{\frac{s_d^2}{F}},$$

*In words:* $s_d^2$ is the sample variance of the fold differences; the extra term $n_{\mathrm{test}}/n_{\mathrm{train}}$
(here about $7.6/30.4 = 0.25$) inflates the uncertainty to account for the overlap between training sets. The statistic
$\bar d/\mathrm{SE}$ is compared with a $t$ distribution with $F-1$ degrees of freedom. It is a heuristic, not an exact
test, but it is far more honest than the naive version.

In [ ]:
# ── 6.1 Paired comparison of the methods across the same outer folds ──
def corrected_paired_t(d, n_train, n_test):
    """Nadeau–Bengio corrected resampled t-test for paired CV differences d. Returns (mean, SE_naive, SE_corr, p)."""
    F, var = len(d), np.var(d, ddof=1)
    se_naive, se_corr = np.sqrt(var / F), np.sqrt((1 / F + n_test / n_train) * var)
    p = 2 * stats.t.sf(abs(d.mean()) / se_corr, F - 1) if se_corr > 0 else np.nan
    return d.mean(), se_naive, se_corr, p


n_te = np.mean([len(te) for te in NCV["L1-logistic"]["test_idx"]])
n_tr = len(y_dev) - n_te
acc = {name: np.array(r["acc"]) for name, r in NCV.items()}
print(f"{'comparison':>32s}   mean diff   SE naive (p)     SE corrected (p)")
for a, b in itertools.combinations(acc, 2):
    m, se_n, se_c, p = corrected_paired_t(acc[a] - acc[b], n_tr, n_te)
    p_naive = 2 * stats.t.sf(abs(m) / se_n, len(acc[a]) - 1) if se_n > 0 else np.nan
    print(f"{a + ' − ' + b:>32s}   {m:+9.3f}   {se_n:.3f} ({p_naive:.2f})    {se_c:.3f} ({p:.2f})")
for name in acc:
    m, _, se_c, p = corrected_paired_t(acc[name] - base_acc, n_tr, n_te)
    print(f"{name + ' − baseline':>32s}   {m:+9.3f}   {'':15s}  {se_c:.3f} ({p:.1e})")

🔍 **What just happened?** Every method beats the baseline by roughly 20–30 percentage points with small corrected
p-values (bottom rows) — the first success criterion of the contract is met. Between the methods, the largest
difference (RBF SVM vs L1-logistic, about 8 points) is only about 1.5 corrected standard errors, and none of the
corrected p-values is small (top rows). **On 38 patients, the data cannot rank these three methods.** Now look at the
naive column: its standard errors are about half as large, and for SVM vs L1-logistic the naive p-value drops to about
0.02 — the naive test would have "found" a winner that the overlap between training folds does not justify.

✏️ **Your turn.** Change `n_repeats` in `nested_cv` from 2 to 5 (it takes about 2.5× longer). Do the mean accuracies
change? Does the corrected p-value for the closest pair become small? Why does repeating CV *not* create new information
about new patients?

### 6.2 Can the predicted probabilities be trusted? Calibration

A classifier that says "P(AML) = 0.9" should be right about 90 % of the time when it says so — it is **calibrated**.
Logistic regression models $P(Y\mid X)$ directly (Module 09); a random forest outputs the fraction of trees voting AML,
which is a score rather than a probability; the SVM outputs a signed distance to the boundary and no probability at all
(Module 11). A standard summary is the **Brier score**,

$$\mathrm{BS} = \frac{1}{N}\sum_{i=1}^{N}\big(\hat p_i - y_i\big)^2 ,$$

*In words:* the mean squared difference between the predicted probability and the 0/1 outcome — 0 is perfect, and
always predicting the base rate $\bar y$ gives $\bar y(1-\bar y)$ (here about 0.21). We use the out-of-fold probabilities
from the nested CV (each patient predicted once per repeat by a model that never saw them).

In [ ]:
# ── 6.2 Brier scores and a reliability diagram from out-of-fold probabilities ──
base_rate = y_dev.mean()
print(f"Brier score of always predicting the base rate {base_rate:.2f}: {base_rate * (1 - base_rate):.3f}")
fig, ax = plt.subplots(figsize=(4.6, 3.8))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
bins = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])
for name, col in [("L1-logistic", "tab:blue"), ("Random forest", "tab:green")]:
    p_oof = NCV[name]["proba"].ravel()
    y_rep = np.tile(y_dev, NCV[name]["proba"].shape[0])
    print(f"{name:>14s}: Brier = {np.mean((p_oof - y_rep) ** 2):.3f}   "
          f"(mean P(AML) for true AML {p_oof[y_rep == 1].mean():.2f}, for true ALL {p_oof[y_rep == 0].mean():.2f})")
    which = np.clip(np.digitize(p_oof, bins) - 1, 0, len(bins) - 2)
    xs = [p_oof[which == b].mean() for b in range(len(bins) - 1) if np.any(which == b)]
    ys = [y_rep[which == b].mean() for b in range(len(bins) - 1) if np.any(which == b)]
    ns = [np.sum(which == b) for b in range(len(bins) - 1) if np.any(which == b)]
    ax.plot(xs, ys, "o-", color=col, label=name)
    for x_, y_, n_ in zip(xs, ys, ns):
        ax.annotate(str(n_), (x_, y_), textcoords="offset points", xytext=(4, -10), fontsize=7, color=col)
ax.set_xlabel("predicted P(AML) (bin average)"); ax.set_ylabel("observed fraction AML")
ax.set_title("Reliability of out-of-fold probabilities\n(numbers: patients per bin, 2 repeats pooled)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Both probabilistic outputs have Brier scores far below the base-rate value (printout), so
their probabilities carry real information. The reliability diagram is coarse — with 76 pooled predictions, most fall
in the lowest bins and the middle bins hold only 5–10 predictions each (numbers on the plot), so single points are very
uncertain. Look at the *direction* of the deviations: the random forest's curve lies **above** the diagonal for
predicted values of about 0.4–0.9 — when its vote fraction says 50–70 %, the patient almost always has AML. That
**under-confidence** is typical of vote fractions, which are pulled towards 0.5 by trees that disagree. The logistic
model's points scatter on both sides of the diagonal, as expected from noise with so few patients per bin. For a
diagnostic use, calibration would have to be checked on far more patients.

✏️ **Your turn.** Give the SVM probabilities with Platt scaling (a logistic regression fitted to its decision values on
inner folds): replace its classifier by `sklearn.calibration.CalibratedClassifierCV(SVC(...), cv=3)` in `CANDIDATES`
and compare its Brier score with the other two.

---
## 7 · Interpretation and stability: which genes, and how sure are we?

> 📍 **Where we are.** Prediction is only half of the biological question ("…and which genes carry the signal?"). Each
> outer fold produced its own fitted L1-logistic model and random forest; comparing them tells us how **stable** the
> selected genes are — a property every lasso selection should be checked for (Module 07).

In [ ]:
# ── 7.1 Genes selected by the L1-logistic model in each outer fold ──
def l1_selected(pipe):
    """Indices (into all genes) of the non-zero L1-logistic coefficients of a fitted pipeline."""
    keep = pipe.named_steps["topgenesbyt"].keep_
    coef = pipe.named_steps["logisticregression"].coef_.ravel()
    return set(keep[coef != 0])


sel = [l1_selected(m) for m in NCV["L1-logistic"]["models"]]
n_folds = len(sel)
print("genes selected per outer fold:", [len(s) for s in sel])
jac = [len(a & b) / len(a | b) for a, b in itertools.combinations(sel, 2) if a | b]
print(f"Jaccard overlap between the gene sets of two folds: median {np.median(jac):.2f} (range {min(jac):.2f}–{max(jac):.2f})")
freq = pd.Series([g for s in sel for g in s]).value_counts()
print(f"{len(freq)} different genes were selected at least once; {np.sum(freq == n_folds)} in all {n_folds} folds; "
      f"{np.sum(freq >= n_folds / 2)} in at least half")
print("number of genes selected in exactly f of the folds:",
      {int(f): int(n) for f, n in freq.value_counts().sort_index().items()})
imp = np.zeros(X_dev.shape[1])                              # random-forest importances, averaged over folds
for m in NCV["Random forest"]["models"]:
    imp[m.named_steps["topgenesbyt"].keep_] += m.named_steps["randomforestclassifier"].feature_importances_ / n_folds
rf_top = set(np.argsort(imp)[::-1][:20])
print(f"overlap of the 20 most important random-forest genes with genes chosen by L1 in ≥ half the folds: "
      f"{len(rf_top & set(freq[freq >= n_folds / 2].index))}")

In [ ]:
# ── 7.2 The most stable genes: selection frequency and expression by class ──
top_stable = freq.index[:12]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw=dict(width_ratios=[1.6, 1]))
axes[0].bar(range(len(top_stable)), freq.values[:12] / n_folds,
            color=["tab:purple" if g in rf_top else "tab:blue" for g in top_stable])
axes[0].set_xticks(range(len(top_stable)), genes[top_stable], rotation=60, ha="right", fontsize=7)
axes[0].set_ylabel("fraction of outer folds selecting the gene")
axes[0].set_title("L1-logistic: most frequently selected genes (purple: also in RF top 20)")
g0 = top_stable[0]
for c, lab in enumerate(["ALL", "AML"]):
    v = X_dev[y_dev == c, g0]
    axes[1].scatter(np.full(len(v), c) + rng.uniform(-0.1, 0.1, len(v)), v, s=18, color=CLASS_COLORS[c])
axes[1].set_xticks([0, 1], ["ALL", "AML"]); axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylabel("log10 expression"); axes[1].set_title(f"most stable gene: {genes[g0]}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The L1 model keeps between about 10 and 20 of the 200 filtered genes in each fold (counts
printed), and the gene sets of different folds overlap only partly (the Jaccard index: 1 = identical sets, 0 = disjoint;
the median is about 0.2). A small core of genes is chosen in most folds — one in every fold — while most of the 54
genes ever selected appear only once or twice. This is the lasso's well-known behaviour with **correlated
predictors**: when several genes carry nearly the same information, which one enters the model depends on small
changes of the training set (Module 07). The random forest, which spreads importance over many correlated genes,
ranks several of the same genes highly (purple bars), which is mild reassurance that they reflect a real signal. The
most stable gene differs strongly between ALL and AML in the development patients (right panel); with the real data it
is M27891_at, the same probe that had the largest PC1 loading in [Module 14](14_pca_dimensionality_reduction.ipynb) —
an unsupervised and a supervised analysis pointing at the same gene.

**What the models can claim — and what not.**

| ✅ Supported by this analysis | ❌ Not supported |
|---|---|
| Expression profiles of these samples separate ALL from AML far better than the baseline, under the stated protocol. | That any selected gene *causes* or *drives* AML — selection reflects predictive association, not mechanism. |
| A small, partly stable set of genes is sufficient for that separation. | That the selected set is *the* set of relevant genes — correlated genes are interchangeable for prediction. |
| Three very different model families reach similar accuracy. | That one method is better for leukaemia in general (38 patients cannot rank them; Section 6). |
| The genes are hypotheses to check in the literature and in new samples. | Clinical use: that would need many more patients, centres and a prospective evaluation. |

✏️ **Your turn.** The probe IDs are Affymetrix HU6800 identifiers (mostly GenBank accessions). Look up the three most
stable ones in a database (NCBI Gene / GenBank) and write one sentence each on whether their known function makes sense
for myeloid vs lymphoid cells. Then compare them with the top PC1 loadings of [Module 14](14_pca_dimensionality_reduction.ipynb).

---
## 8 · Unsealing: the final models on the 34 independent patients

> 📍 **Where we are.** Everything so far used the 38 development patients. Now — once — we fit each method on all of
> them, with hyper-parameters chosen by a final 5-fold CV on the development set, and predict the sealed test set.

For a test set of $n$ patients with $k$ correct, the accuracy $k/n$ gets an exact (**Clopper–Pearson**) 95 % confidence
interval from the binomial distribution. To compare two methods on the *same* patients we use two paired tools:

* a **bootstrap**: resample the 34 patients with replacement 5,000 times and recompute the accuracy difference; the
  middle 95 % of those differences is a confidence interval;
* **McNemar's exact test**: only patients on which the two methods *disagree* carry information. If $b$ patients are
  right by method A only and $c$ by method B only, then under "no difference" $b \sim \mathrm{Binomial}(b+c, \tfrac12)$.

*In words:* both ask whether the disagreements lean consistently towards one method more than chance would allow.

In [ ]:
# ── 8.1 Final tuning on all development patients, then ONE prediction of the sealed test set ──
final_folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED + 999).split(X_dev, y_dev))
FINAL, PRED = {}, {}
for name, cands in CANDIDATES.items():
    cv_acc = [np.mean([accuracy(y_dev[b], clone(m).fit(X_dev[a], y_dev[a]).predict(X_dev[b])) for a, b in final_folds])
              for _, m in cands]
    best = int(np.argmax(cv_acc))
    FINAL[name] = clone(cands[best][1]).fit(X_dev, y_dev)  # refit on all 38 development patients
    PRED[name] = FINAL[name].predict(X_test)                # the single look at the sealed test set
    print(f"{name:>14s}: chosen {cands[best][0]:<28s} (dev CV accuracy {cv_acc[best]:.3f})")
PRED["Baseline (always ALL)"] = np.zeros(len(y_test), int)

print(f"\n{'method':>22s}  correct   accuracy [95 % CI]     AML precision  AML recall   nested-CV estimate")
for name, pred in PRED.items():
    k, n = int(np.sum(pred == y_test)), len(y_test)
    ci = stats.binomtest(k, n).proportion_ci(confidence_level=0.95, method="exact")
    prec, rec, _ = precision_recall_f1(y_test, pred)
    est = np.mean(NCV[name]["acc"]) if name in NCV else base_acc.mean()
    print(f"{name:>22s}  {k:2d}/{n}    {k / n:.3f} [{ci.low:.3f}, {ci.high:.3f}]   {prec:13.3f}  {rec:10.3f}   {est:.3f}")

In [ ]:
# ── 8.2 Confusion matrices, bootstrap differences and McNemar's test ──
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, name in zip(axes, CANDIDATES):
    c = confusion_counts(y_test, PRED[name])
    mat = np.array([[c["TN"], c["FP"]], [c["FN"], c["TP"]]])
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=len(y_test))
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=12, color="k" if v < 15 else "w")
    ax.set_xticks([0, 1], ["ALL", "AML"]); ax.set_yticks([0, 1], ["ALL", "AML"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{name}: test set")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED + 8)
correct = {name: (PRED[name] == y_test).astype(float) for name in CANDIDATES}
boot = rng.integers(0, len(y_test), size=(5000, len(y_test)))   # 5,000 resamples of the 34 test patients
SHORT = {"L1-logistic": "L1", "Random forest": "RF", "RBF SVM": "SVM"}
print("misclassified test patients (positions 0–33):", {n: [int(i) for i in np.flatnonzero(c == 0)] for n, c in correct.items()})
for a, b in itertools.combinations(CANDIDATES, 2):
    diff = correct[a][boot].mean(axis=1) - correct[b][boot].mean(axis=1)
    lo, hi = np.percentile(diff, [2.5, 97.5])
    only_a, only_b = int(np.sum((correct[a] == 1) & (correct[b] == 0))), int(np.sum((correct[a] == 0) & (correct[b] == 1)))
    p_mc = stats.binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    print(f"{a} − {b}: Δacc = {correct[a].mean() - correct[b].mean():+.3f}, bootstrap 95 % CI [{lo:+.3f}, {hi:+.3f}]; "
          f"right only by {SHORT[a]}: {only_a}, only by {SHORT[b]}: {only_b}, McNemar p = {p_mc:.2f}")

🔍 **What just happened?** On patients collected separately from the development set, all three methods again beat the
baseline (which is right only for the 20 ALL patients, 59 %), and they make only a few errors each. The confidence
intervals are wide — with 34 patients, a single error moves accuracy by 3 percentage points and the lower ends of the
intervals sit well below the point estimates. The methods' errors partly fall on the **same patients** (the printed
positions), a hint that those samples are genuinely hard (or atypical) rather than that one method is flawed. Every
bootstrap interval for a difference includes 0 and every McNemar p-value is large: **the test set cannot rank the
methods either.** (For the forest vs the SVM the interval ends exactly at 0: the two disagree on a single patient, so
every bootstrap resample gives a difference of either 0 or a negative value — a reminder that with a handful of
discordant patients, bootstrap intervals are coarse.)

Compare each method's test accuracy with its nested-CV estimate (last column of the table above). The RBF SVM, perfect
in all ten outer folds, misclassified two test patients; the L1-logistic model, the lowest in CV, misclassified only
one. Both observations sit comfortably inside the uncertainty: a CV estimate of 100 % from folds of 7–8 patients does
not mean "never wrong", and a test accuracy *above* the CV estimate is luck as much as merit. A test accuracy far
*below* the estimate would have warned of a shift between development and test patients — a real possibility here,
since the independent set was collected separately and includes blood as well as bone-marrow samples.

**Conclusion for the report.** *Expression profiles distinguish ALL from AML with high accuracy under a nested-CV
protocol and on an independent test set; L1-logistic regression, a random forest and an RBF SVM are statistically
indistinguishable on these data; a small, partly stable gene set suffices; probabilities from the logistic model are
informative but their calibration cannot be assessed precisely with this sample size.* That is a modest, defensible and
useful result — which is what a good project reports.

---
## 9 · Alternative mini-project: learning a signalling network from single cells

> 📍 **Where we are.** Not every project is a classifier. This section is a complete second template — one data set,
> three *structure-learning* methods from Part D, one evaluation — for students who prefer networks.

🧫 **Data.** Sachs *et al.* (2005) measured 11 phosphorylated proteins and phospholipids (raf, mek, plc, pip2, pip3,
erk, akt, pka, pkc, p38, jnk) in thousands of individual human T cells by multiparameter flow cytometry, under several
perturbations, and learned a Bayesian network that recovered most known signalling connections. We use only the
**unperturbed** cells (`observational_only=True`): the discretised version (low/medium/high) for the two
Bayesian-network methods and the continuous (log-scale) version for the Gaussian graphical model.

**Evaluation contract.** Unit: a cell (⚠️ cells from the same donor and experiment are not fully independent — a
limitation to state). Truth: the 20-edge consensus network distributed with the data (`SACHS_CONSENSUS_EDGES`), compared
as an **undirected skeleton**. Metrics: precision (fraction of learned edges that are in the consensus) and recall
(fraction of consensus edges found). Baseline: a random graph — its expected precision is the consensus density,
$20/\binom{11}{2} = 20/55 \approx 0.36$.

**Three methods** (Modules 17–18): a **Chow–Liu tree** (the maximum-likelihood tree: maximum spanning tree of pairwise
mutual information), **score-based hill climbing** over DAGs with the BIC score, and the **graphical lasso** (sparse
precision matrix of a Gaussian).

In [ ]:
# ── 9.1 Load both versions of the data; the consensus skeleton and a scoring function ──
sachs_d = load_sachs("discrete", observational_only=True)
sachs_c = load_sachs("continuous", observational_only=True)
print("Source:", sachs_d["source"], "|", sachs_c["source"])
D = sachs_d["data"][SACHS_PROTEINS].to_numpy()            # cells × 11, values 0/1/2
C_cont = sachs_c["data"][SACHS_PROTEINS].to_numpy()        # cells × 11, log-scale intensities
P = len(SACHS_PROTEINS)
print("discrete:", D.shape, "| continuous:", C_cont.shape)
TRUTH = {frozenset(e) for e in SACHS_CONSENSUS_EDGES}


def score_skeleton(edges):
    """Precision, recall and number of edges of an (undirected view of a) learned graph vs the consensus."""
    learned = {frozenset((SACHS_PROTEINS[i], SACHS_PROTEINS[j])) for i, j in edges}
    tp = len(learned & TRUTH)
    return (tp / len(learned) if learned else 0.0), tp / len(TRUTH), len(learned)


print(f"consensus: {len(TRUTH)} edges among {P * (P - 1) // 2} protein pairs → random-graph precision ≈ "
      f"{len(TRUTH) / (P * (P - 1) / 2):.2f}")

In [ ]:
# ── 9.2 Method 1 — Chow–Liu tree: mutual information + maximum spanning tree (Kruskal) ──
def mutual_information(a, b, levels=3):
    """Empirical mutual information (nats) between two discrete variables coded 0 … levels−1."""
    joint = np.zeros((levels, levels))
    np.add.at(joint, (a, b), 1)
    joint /= joint.sum()
    outer = joint.sum(axis=1, keepdims=True) @ joint.sum(axis=0, keepdims=True)
    nz = joint > 0
    return float(np.sum(joint[nz] * np.log(joint[nz] / outer[nz])))


def chow_liu_edges(data):
    """Edges of the maximum-weight spanning tree of the pairwise mutual-information matrix (Kruskal + union–find)."""
    p = data.shape[1]
    weights = sorted(((mutual_information(data[:, i], data[:, j]), i, j) for i, j in itertools.combinations(range(p), 2)),
                     reverse=True)
    root = list(range(p))

    def find(v):
        while root[v] != v:
            v = root[v]
        return v
    tree = []
    for _, i, j in weights:                                # strongest pairs first; skip edges that close a cycle
        if find(i) != find(j):
            root[find(i)] = find(j)
            tree.append((i, j))
    return tree


CL = chow_liu_edges(D)
print("Chow–Liu tree:", ", ".join(f"{SACHS_PROTEINS[i]}–{SACHS_PROTEINS[j]}" for i, j in CL))
print("precision %.2f, recall %.2f, %d edges" % score_skeleton(CL))

In [ ]:
# ── 9.3 Method 2 — BIC score of a discrete Bayesian network (decomposes over families) ──
_FAMILY_CACHE = {}


def family_bic(i, parents, data=D, levels=3):
    """BIC contribution of node i with the given parent tuple: log-likelihood − (#free parameters / 2)·log N."""
    key = (i, parents)
    if key not in _FAMILY_CACHE:
        N = len(data)
        cfg = (np.ravel_multi_index(data[:, list(parents)].T, (levels,) * len(parents)) if parents
               else np.zeros(N, int))
        counts = np.zeros((levels ** len(parents), levels))
        np.add.at(counts, (cfg, data[:, i]), 1)            # counts[parent configuration, value of node i]
        totals = counts.sum(axis=1, keepdims=True)
        nz = counts > 0
        loglik = np.sum(counts[nz] * np.log((counts / np.where(totals > 0, totals, 1))[nz]))
        n_free = levels ** len(parents) * (levels - 1)     # one CPT row per parent configuration
        _FAMILY_CACHE[key] = loglik - 0.5 * n_free * np.log(N)
    return _FAMILY_CACHE[key]


def has_cycle(parents):
    """True if the parent sets {node: tuple of parents} contain a directed cycle (depth-first search)."""
    state = {}                                             # 1 = on the current path, 2 = finished

    def visit(v):
        state[v] = 1
        for u in parents[v]:
            if state.get(u) == 1 or (u not in state and visit(u)):
                return True
        state[v] = 2
        return False
    return any(v not in state and visit(v) for v in parents)

In [ ]:
# ── 9.4 Hill climbing over DAGs: add, delete or reverse one edge at a time (slides 008: 169–172) ──
def hill_climb(p, max_parents=3):
    """Greedy BIC hill climbing from the empty graph; returns (parent sets, list of (move, from, to, gain))."""
    parents, history = {i: () for i in range(p)}, []
    while True:
        best = (1e-9, None)
        for j, i in itertools.permutations(range(p), 2):   # consider the edge j → i
            cur_i = parents[i]
            if j in cur_i:                                 # delete j → i, or reverse it to i → j
                moves = [("delete", {i: tuple(sorted(set(cur_i) - {j}))})]
                if len(parents[j]) < max_parents:
                    moves.append(("reverse", {i: tuple(sorted(set(cur_i) - {j})),
                                              j: tuple(sorted(set(parents[j]) | {i}))}))
            elif i not in parents[j] and len(cur_i) < max_parents:
                moves = [("add", {i: tuple(sorted(set(cur_i) | {j}))})]
            else:
                moves = []
            for kind, change in moves:
                trial = {**parents, **change}
                if kind != "delete" and has_cycle(trial):
                    continue                               # adding/reversing must keep the graph acyclic
                gain = sum(family_bic(v, trial[v]) - family_bic(v, parents[v]) for v in change)
                if gain > best[0]:
                    best = (gain, (kind, j, i, trial))
        if best[1] is None:                                # no move improves the score: a local maximum
            return parents, history
        kind, j, i, parents = best[1]
        history.append((kind, SACHS_PROTEINS[j], SACHS_PROTEINS[i], round(float(best[0]), 1)))


HC_PARENTS, HC_HISTORY = hill_climb(P)
HC = [(j, i) for i in range(P) for j in HC_PARENTS[i]]     # directed edges j → i
print("moves:", HC_HISTORY)
print("precision %.2f, recall %.2f, %d edges (skeleton)" % score_skeleton(HC))
consensus_dir = set(SACHS_CONSENSUS_EDGES)
same = sum((SACHS_PROTEINS[j], SACHS_PROTEINS[i]) in consensus_dir for j, i in HC)
rev = sum((SACHS_PROTEINS[i], SACHS_PROTEINS[j]) in consensus_dir for j, i in HC)
print(f"of the learned edges that are in the consensus skeleton: {same} point the same way, {rev} are reversed")

In [ ]:
# ── 9.5 Method 3 — graphical lasso on the continuous data, for several penalties ──
Z_cont = standardize(C_cont)[0]
GL = {}
print(" alpha   edges   precision   recall")
for alpha in (0.02, 0.05, 0.1, 0.2):
    theta = GraphicalLasso(alpha=alpha, max_iter=500).fit(Z_cont).precision_
    GL[alpha] = [(i, j) for i, j in itertools.combinations(range(P), 2) if abs(theta[i, j]) > 1e-6]
    prec, rec, n_e = score_skeleton(GL[alpha])
    print(f"{alpha:6.2f}   {n_e:5d}   {prec:9.2f}   {rec:6.2f}")

In [ ]:
# ── 9.6 The three learned networks next to the consensus ──
angles = np.linspace(0, 2 * np.pi, P, endpoint=False) + np.pi / 2
ring = ["raf", "mek", "erk", "akt", "pip3", "pip2", "plc", "pkc", "p38", "jnk", "pka"]   # order on the circle
pos = {name: (0.5 + 0.42 * np.cos(a), 0.5 + 0.42 * np.sin(a)) for name, a in zip(ring, angles)}
ALPHA_SHOW = 0.1
graphs = [("consensus (20 edges)", [tuple(e) for e in SACHS_CONSENSUS_EDGES], True),
          ("Chow–Liu tree", [(SACHS_PROTEINS[i], SACHS_PROTEINS[j]) for i, j in CL], False),
          ("BIC hill climbing", [(SACHS_PROTEINS[j], SACHS_PROTEINS[i]) for j, i in HC], True),
          (f"graphical lasso (α={ALPHA_SHOW})", [(SACHS_PROTEINS[i], SACHS_PROTEINS[j]) for i, j in GL[ALPHA_SHOW]], False)]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, (title, edges, directed) in zip(axes, graphs):
    hits = [e for e in edges if frozenset(e) in TRUTH]
    misses = [e for e in edges if frozenset(e) not in TRUTH]
    draw_graph(hits, pos, directed=directed, ax=ax, radius=0.075)
    for u, v in misses:                                    # edges not in the consensus: red dashed
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], ls="--", color="tab:red", lw=1.2, zorder=1)
    ax.set_title(f"{title}\n{len(hits)} in consensus, {len(misses)} not (red dashed)", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.7 Stability: how often does each Chow–Liu edge reappear in bootstrap resamples of the cells? ──
rng = np.random.default_rng(SEED + 9)
B = 30
edge_count = {}
for _ in range(B):
    boot_cells = D[rng.integers(0, len(D), len(D))]
    for i, j in chow_liu_edges(boot_cells):
        e = frozenset((SACHS_PROTEINS[i], SACHS_PROTEINS[j]))
        edge_count[e] = edge_count.get(e, 0) + 1
print(f"{'edge':>12s}  bootstrap frequency  in consensus?")
for e, c in sorted(edge_count.items(), key=lambda kv: -kv[1]):
    if c >= 0.2 * B:
        print(f"{'–'.join(sorted(e)):>12s}  {c / B:19.2f}  {'yes' if e in TRUTH else 'no'}")

🔍 **What just happened?** All three methods find networks whose edges are mostly in the consensus — their
precisions (printed) are well above the random-graph value of 0.36 — but each finds only part of it (recall well below
1). The **Chow–Liu tree** is forced to have exactly $11-1=10$ edges and no loops, whereas the consensus network is full
of triangles (e.g. pka → raf → mek together with pka → mek), so a tree can never recover more than part of it. **BIC hill climbing** stops early: with three levels per variable, every extra parent
multiplies the number of CPT parameters by 3, and the $\frac{k}{2}\log N$ penalty of BIC (Module 23) keeps the graph
sparse. And the *directions* it learns mostly disagree with the consensus (printed counts): from observational data a
Bayesian network is identifiable only up to its equivalence class, so arrows often cannot be trusted (Module 17) —
Sachs *et al.* needed their interventions to orient edges. The **graphical lasso** trades precision for recall as the
penalty $\alpha$ decreases (Module 18's path). The bootstrap shows which Chow–Liu edges are robust: the edges that reappear in (nearly) every resample are all in
the consensus, whereas the edges that come and go are mostly not — stability is a useful, label-free warning sign.

⚠️ **Caveats for the report.** (1) A learned edge is a *statistical dependence* in these cells, not proof of a
biochemical interaction or of its direction. (2) The "consensus" is itself a summary of the literature, possibly
incomplete. (3) Discretisation into three levels and the Gaussian assumption are modelling choices that affect the
result. (4) Cells are not independent replicates of the biology; donors or experiments would be.

✏️ **Your turn — make it your project.** (a) Add random restarts to the hill climber (start from random sparse DAGs)
and keep the best score. (b) Choose $\alpha$ for the graphical lasso by cross-validated likelihood
(`sklearn.covariance.GraphicalLassoCV`) instead of by eye. (c) Use the perturbed cells too (`observational_only=False`)
— how would you use the knowledge of which protein was inhibited?

### 📑 Slide connection — `008_BN_mz_021825 & 009_GGM_mz_030925`, slides 008: 157–172; 009: 49–58

- **What the slides show.** Learning a Bayesian-network structure is an open problem in general; tree structures can be learned exactly (Chow–Liu), general DAGs by heuristic search or exponential-time exact methods (008 slide 157). Chow–Liu minimises the KL divergence to the true distribution, which amounts to a maximum spanning tree of pairwise mutual information (slides 158–168). Hill climbing adds, deletes or reverses an edge, rejects cycles, keeps the best-scoring neighbour and stops when no move improves; local maxima and plateaus call for random restarts, TABU search or simulated annealing (slides 169–172). Sparse GGMs via an L1 penalty on the precision matrix — the graphical lasso (009 slides 49–57); inference and learning are efficient for Gaussian, not for discrete, undirected models (slide 58).
- **What this notebook adds.** All three structure learners run on the real Sachs cells and are scored against the consensus skeleton with precision and recall, a random-graph baseline, a direction check and a bootstrap of edge stability.
- **Exam tip.** Know why Chow–Liu always returns $n-1$ edges, why BIC favours sparse graphs, and why observational data cannot orient every edge.

---
## 10 · Writing it up: results table, report template, peer review and presentation

> 📍 **Where we are.** The analysis is done. A project is finished only when someone else can understand, check and
> reproduce it. The next cell assembles the numbers of the worked example into the kind of table a report needs.

In [ ]:
# ── 10.1 One table with every number the report needs ──
table = []
for name in list(CANDIDATES) + ["Baseline (always ALL)"]:
    k = int(np.sum(PRED[name] == y_test))
    ci = stats.binomtest(k, len(y_test)).proportion_ci(confidence_level=0.95, method="exact")
    cv = np.array(NCV[name]["acc"]) if name in NCV else base_acc
    table.append({"method": name, "nested-CV acc (mean ± SD, 10 folds)": f"{cv.mean():.3f} ± {cv.std(ddof=1):.3f}",
                  "test acc (34 patients)": f"{k / len(y_test):.3f}",
                  "test 95 % CI": f"[{ci.low:.2f}, {ci.high:.2f}]",
                  "AML recall (test)": f"{precision_recall_f1(y_test, PRED[name])[1]:.2f}"})
results_table = pd.DataFrame(table).set_index("method")
print(results_table.to_string())
print(f"\nSoftware: Python {sys.version.split()[0]}, NumPy {np.__version__}, scikit-learn {sklearn.__version__}; SEED = {SEED}")

### 📝 Report template (copy into your own document)

```text
TITLE — a claim-shaped title (e.g. "Expression profiles separate ALL from AML; three classifiers are indistinguishable")
Team members and contributions (who did what)

ABSTRACT (≤ 150 words): question · data · methods · main result WITH uncertainty · main limitation

1. QUESTION AND BIOLOGICAL CONTEXT
   Why does the question matter? What would a biologist or clinician do with the answer?
2. DATA
   Source and citation · N samples × J features · unit of replication · preprocessing · class balance · known batches
3. EVALUATION CONTRACT (written before modelling — paste the dictionary of Section 2)
   Target · split (and why) · metric(s) · baseline · success criterion
4. METHODS (for each of the three)
   Model and family (probabilistic? generative/discriminative?) · how parameters are estimated (MLE/MAP/other)
   · hyper-parameters and the grid searched · preprocessing done INSIDE the training folds
   Protocol: nested CV (outer/inner folds, repeats), same folds for all methods, final refit, single test evaluation
5. RESULTS
   Exploratory figure (one) · results table with CIs (like 10.1) · paired comparison (corrected) · calibration
6. INTERPRETATION AND STABILITY
   Selected features and their stability · agreement between methods · what IS and IS NOT supported (table)
7. LIMITATIONS AND THREATS TO VALIDITY
   Sample size · independence · distribution shift · multiple comparisons · what you tuned and on which data
8. CONCLUSION AND NEXT EXPERIMENT
   One paragraph; the experiment or data set that would most change your conclusion
REPRODUCIBILITY
   Code (notebook), software versions, random seeds, data access
REFERENCES
```

### 🔎 Peer-review rubric (for proposals and reports)

| Criterion | 3 — excellent | 2 — adequate | 1 — needs work |
|---|---|---|---|
| **Question** | specific, biologically motivated, answerable with the data | clear but generic | a method demo without a question |
| **Data & unit of replication** | source, size, preprocessing and independence stated and respected in the splits | described, independence not discussed | unclear; risk of leakage between related samples |
| **Evaluation contract** | fixed in advance; baseline; metric suits the question | present but partly decided after seeing results | missing; test data used for choices |
| **Methods** | three methods from different families, justified, correctly described, equal tuning budget | appropriate but unequal tuning or thin justification | inappropriate, or tuned on the test set |
| **Uncertainty** | CIs on all key numbers, paired comparison, honest about ties | some uncertainty given | point estimates only; winners declared from noise |
| **Interpretation** | claims calibrated to the evidence; stability checked; limitations explicit | mostly calibrated | causal or clinical over-claiming |
| **Reproducibility** | code, seeds, versions, data access; reruns give the same numbers | mostly reproducible | cannot be rerun |
| **Clarity** | one clear story; figures with labelled axes and units; concise | understandable | hard to follow |

**Writing a review.** Start with a two-sentence summary of what the authors did (proving you understood it); then the
two or three most important issues, each with a concrete suggestion; then minor points. Be specific, kind and useful —
review the work, not the people.

### 🎤 Presentation checklist (≈ 10 minutes)

- [ ] Slide 1: the biological question in one sentence and why it matters
- [ ] The data: where from, $N \times J$, unit of replication, one exploratory figure
- [ ] The evaluation contract: split, metric, baseline
- [ ] The three methods in one line each (family, key hyper-parameters)
- [ ] The main result **with uncertainty** (one table or one figure) and the baseline
- [ ] One interpretation slide (features, stability) and one limitations slide
- [ ] The take-home message and the next experiment
- [ ] Every axis labelled; every number has a unit or a denominator

---
## 11 · The course in one page

> 📍 **Where we are.** The end of BiologyToML. This section condenses 15 lectures and 24 modules into one map, three
> recurring ideas and a self-assessment you can use to prepare for the 02-620 exams.

### The method map (task × kind of model)

| Task | Non-probabilistic | Generative (models $P(X, Y)$ or $P(X)$) | Discriminative (models $P(Y \mid X)$) |
|---|---|---|---|
| **Regression** | — | — | linear regression, ridge, lasso ([06](06_linear_regression_gwas.ipynb), [07](07_ridge_lasso_sparse_regression.ipynb)) |
| **Classification** | k-NN ([08](08_knn_naive_bayes.ipynb)), decision trees & random forests ([10](10_decision_trees_random_forests.ipynb)), SVM ([11](11_support_vector_machines.ipynb)) | naïve Bayes ([08](08_knn_naive_bayes.ipynb)) | logistic regression ([09](09_logistic_regression.ipynb)), neural networks ([22](22_neural_networks.ipynb)) |
| **Clustering** | k-means, hierarchical ([12](12_clustering_kmeans_hierarchical.ipynb)) | mixture models with EM ([13](13_mixture_models_em.ipynb)) | — |
| **Dimensionality reduction** | PCA ([14](14_pca_dimensionality_reduction.ipynb)), autoencoders ([22](22_neural_networks.ipynb)) | probabilistic PCA, variational autoencoders ([22](22_neural_networks.ipynb)) | — |
| **Networks and sequences** | — | Bayesian networks ([15](15_bayesian_networks_representation.ipynb)–[17](17_bayesian_networks_inference_learning.ipynb)), GGMs ([18](18_gaussian_graphical_models.ipynb)), HMMs/DBNs ([19](19_hmm_inference.ipynb)–[20](20_hmm_learning.ipynb)) | — |

Across the table: **estimation** by MLE, MAP or full Bayes ([05](05_likelihood_mle_map.ipynb),
[21](21_bayesian_learning_mcmc.ipynb)) and **model selection** by cross-validation, regularisation or BIC
([23](23_model_selection.ipynb)); the foundations are in Modules [00](00_welcome_ml_for_biology.ipynb)–[05](05_likelihood_mle_map.ipynb).

### The three ideas that recur in every lecture

1. **The two-step template.** Step 1: specify a probability model ($P(Y\mid X;\theta)$ or $P(X;\theta)$); Step 2: estimate
   $\theta$ by MLE, MAP or full Bayes. *Example:* logistic regression with a Gaussian prior is MAP with an L2 penalty
   ([09](09_logistic_regression.ipynb)); in this capstone the L1-logistic model is MAP with a Laplace prior. The
   non-probabilistic exceptions are flagged as such and paired with probabilistic cousins: k-means ↔ Gaussian mixtures,
   SVM ↔ logistic regression, PCA ↔ probabilistic PCA; single trees are paired with forests that average away their
   variance.
2. **Conditional independence buys tractability.** *Examples:* naïve Bayes on $n$ genotyped loci needs $2\cdot n\cdot 2+1$
   instead of $(3^n-1)\cdot 2+1$ parameters ([08](08_knn_naive_bayes.ipynb)); the alarm network needs 10 instead of 31
   ([15](15_bayesian_networks_representation.ipynb)); an HMM's chain lets the forward algorithm compute $P(O)$ in $O(n^2T)$ instead of summing
   over all $n^T$ state paths ([19](19_hmm_inference.ipynb)); a sparse precision matrix is a sparse network ([18](18_gaussian_graphical_models.ipynb)).
   In this capstone, the Chow–Liu tree and BIC hill climbing are two ways of *choosing* those independences.
3. **Latent variables ⇒ EM.** MLE is easy when everything is observed and inference is easy when parameters are known;
   when neither holds, alternate. *Examples:* Gaussian mixtures (responsibilities, [13](13_mixture_models_em.ipynb)),
   Baum–Welch for HMMs (expected counts, [20](20_hmm_learning.ipynb)), k-means as the hard-assignment limit
   ([12](12_clustering_kmeans_hierarchical.ipynb)), and Gibbs sampling as the sampling sibling ([21](21_bayesian_learning_mcmc.ipynb)).

### ✅ Self-assessment: every deck's "What you should know", linked to the modules

Tick a box only if you could explain the point to a classmate *and* reproduce the key derivation or computation without
notes. The items paraphrase each deck's own summary or "What you should know" slide, completed by a few points the
lectures stress elsewhere (for example k-NN and precision/recall, which the `003`/`004` summary slide omits, and the
fact that SVMs need model selection).

**`001_intro`** — [00](00_welcome_ml_for_biology.ipynb), [05](05_likelihood_mle_map.ipynb)
- [ ] Supervised learning learns $f: X\to Y$, often as $P(Y\mid X)$ to represent uncertainty; unsupervised learning learns $P(X)$ to find patterns.
- [ ] MLE vs MAP: estimate from the data alone vs combine a prior with the likelihood by Bayes' rule.

**`002_regression`** — [06](06_linear_regression_gwas.ipynb), [07](07_ridge_lasso_sparse_regression.ipynb)
- [ ] Linear regression: MLE under Gaussian noise = least squares, $w=(X^\top X)^{-1}X^\top y$.
- [ ] Linear regression with a prior: MAP estimation = ridge regression, needed when $N<J$.
- [ ] The lasso gives sparse estimates of the regression coefficients.

**`003_classification-kNN-NB`** and **`004_classification-LR`** — [08](08_knn_naive_bayes.ipynb), [09](09_logistic_regression.ipynb)
- [ ] k-NN: non-parametric, choice of $K$, cost at test time.
- [ ] Naïve Bayes: model via $P(X\mid Y)$ and $P(Y)$; MLE in closed form (counts), no iterative learning; pseudo-counts.
- [ ] Logistic regression: model $P(Y\mid X)$; MLE has no closed form; iterative learning by gradient ascent; MAP = L2.
- [ ] Generative vs discriminative classifiers; precision and recall.

**`005_decision-tree`** — [10](10_decision_trees_random_forests.ipynb)
- [ ] Greedy top-down learning of decision trees (ID3, C4.5) with information gain.
- [ ] Overfitting and post-pruning to avoid it.
- [ ] Random forests: an ensemble of trees trained on bootstrapped samples with random feature subsets.

**`006_clustering`** — [12](12_clustering_kmeans_hierarchical.ipynb)
- [ ] Clustering as unsupervised learning; the different types of clustering algorithms (k-means, hierarchical with different linkages).
- [ ] The assumptions each makes and what each delivers; unsolved issues: number of clusters, initialisation.

**`007_SVM`** — [11](11_support_vector_machines.ipynb)
- [ ] The maximum-margin principle and the SVM objective; separable and non-separable (soft-margin) cases.
- [ ] The dual formulation and support vectors.
- [ ] The kernel trick and its computational cost — and that SVMs do need model selection ($C$, kernel, $\gamma$).

**`008_BN`** — [15](15_bayesian_networks_representation.ipynb), [16](16_bayesian_networks_independence.ipynb), [17](17_bayesian_networks_inference_learning.ipynb)
- [ ] Representation: a DAG + conditional distributions; Markov blanket and d-separation decode the independence assumptions.
- [ ] Inference: NP-hard in general, efficient for some graphs; variable elimination and stochastic (sampling) methods.
- [ ] Learning: easy for a known graph and fully observed data (MLE, MAP); Chow–Liu for trees; hardest when the graph is unknown.

**`009_GGM`** — [18](18_gaussian_graphical_models.ipynb)
- [ ] Gaussian graphical models are undirected models of continuous variables; zeros of the precision matrix = missing edges.
- [ ] Inference and learning are efficient (graphical lasso for sparsity) — unlike undirected models of discrete variables.

**`010_mixture-model`** — [13](13_mixture_models_em.ipynb)
- [ ] Mixture models as the probabilistic version of k-means (soft assignments).
- [ ] The EM algorithm for learning mixture models; what it guarantees (the likelihood never decreases) and what it does not (a global optimum).

**`011_PCA`** — [14](14_pca_dimensionality_reduction.ipynb)
- [ ] PCA is unsupervised: reduce dimension while preserving the variability of the data.
- [ ] In practice: the eigendecomposition of the covariance; top eigenvalues = directions of most variability; their eigenvectors = the new axes.

**`012_HMM-DBN`** — [19](19_hmm_inference.ipynb), [20](20_hmm_learning.ipynb)
- [ ] Why HMMs (gene finding, speech, phasing); inference by the forward–backward algorithm and the various inference tasks.
- [ ] The Viterbi algorithm for the most probable path.
- [ ] Learning: EM (Baum–Welch) with inference as a subroutine.

**`013_Bayesian-MCMC`** — [21](21_bayesian_learning_mcmc.ipynb)
- [ ] The fully Bayesian approach learns the whole posterior distribution, not a point estimate.
- [ ] Gibbs sampling as a special case of MCMC; burn-in and convergence.

**`014_NN`** — [22](22_neural_networks.ipynb)
- [ ] Various architectures of neural networks (CNNs, graph networks, autoencoders, RNNs).
- [ ] Learning by backpropagation and stochastic gradient descent.

**`015_model-selection`** — [23](23_model_selection.ipynb), and this capstone
- [ ] The bias–variance trade-off.
- [ ] The battle against overfitting: cross-validation, the Bayesian information criterion, regularisation.
- [ ] The Bayes error rate as the limit no classifier can beat.

If a box stays empty, reopen the linked module, re-run its key section, change one parameter and write a lab note.

---
## 📌 Key takeaways

- **A project is a question plus a contract.** Fix the target, unit of replication, split, metric, baseline and success
  criterion *before* fitting anything (`001_intro` slides 9–10; `015` slide 33).
- **Compare methods fairly:** the same folds, the same tuning budget, all preprocessing (including gene selection) inside
  the training folds, and nested CV to separate tuning from evaluation.
- **Every number needs its uncertainty.** Fold-to-fold spread, a corrected paired comparison, exact binomial CIs, a
  bootstrap of the test set and McNemar's test — with 38 + 34 patients they show that three good methods are
  indistinguishable, which is a legitimate result.
- **Use the test set once.** A sealed, separately collected test set checks the CV estimate and reveals shifts between
  data sources.
- **Interpret with stability in mind.** Lasso-selected genes vary between folds when genes are correlated; selected ≠
  causal; learned network edges are dependencies, not proven interactions, and their directions are often not identifiable.
- **Write for a reader.** A report template, a review rubric and a presentation checklist turn an analysis into science
  others can check.
- **The course in one sentence:** specify a probability model, use conditional independence to keep it tractable,
  estimate it by MLE/MAP/Bayes (EM when variables are hidden), and select and evaluate it honestly.

---
## ✅ Checkpoint

1. Why must the gene filter be refitted inside every training fold rather than once on all development patients?
   <details><summary>Answer</summary>Choosing genes by their ALL/AML difference uses the labels. If the held-out patients take part in that choice, the genes are tuned to them and the CV accuracy becomes optimistic (selection bias, Ambroise & McLachlan 2002). Inside the fold, the held-out patients play the role of truly new patients.</details>
2. What does nested CV estimate, and why is an inner loop needed?
   <details><summary>Answer</summary>The performance of the whole *procedure* (including hyper-parameter tuning) on new patients. Without the inner loop, the same folds would both choose the hyper-parameters and score them, so the reported score would be the maximum over settings of noisy estimates — biased upwards.</details>
3. Ten paired fold differences have mean 0.02 and standard deviation 0.08; $n_{test}/n_{train}=0.25$. Compute the naive and the corrected standard errors.
   <details><summary>Answer</summary>Naive: $0.08/\sqrt{10} \approx 0.025$. Corrected: $\sqrt{(1/10+0.25)\cdot0.08^2} = 0.08\sqrt{0.35} \approx 0.047$ — almost twice as large; $t \approx 0.02/0.047 \approx 0.4$, nowhere near significance.</details>
4. Two methods disagree on 4 test patients: method A is right on 3 of them, method B on 1. What does McNemar's exact test say?
   <details><summary>Answer</summary>Under no difference, $b \sim \mathrm{Binomial}(4, 1/2)$. $P(b \ge 3)=5/16$, so the two-sided p-value is $2\cdot 5/16 = 0.625$: no evidence that A is better.</details>
5. A test set has 34 patients and a classifier gets 32 right. Roughly how wide is the 95 % confidence interval, and why does this matter for comparing methods?
   <details><summary>Answer</summary>The exact interval is about [0.80, 0.99] — roughly 19 percentage points wide. Differences of one or two patients between methods are far inside it, so they cannot rank methods.</details>
6. The L1-logistic model selects gene G in 4 of 10 folds. What are two possible reasons, and what should a report say?
   <details><summary>Answer</summary>G may be weakly predictive, or it may be one of several correlated genes that carry the same information, of which the lasso picks one depending on the fold. The report should give the selection frequency, mention the correlated alternatives and not claim G is *the* relevant gene.</details>
7. Why does a Chow–Liu tree on 11 proteins always have 10 edges, and why can a BIC hill climber return fewer?
   <details><summary>Answer</summary>A spanning tree on $n$ nodes has $n-1$ edges by definition. The hill climber adds an edge only if it raises log-likelihood by more than the BIC penalty $\frac{\Delta k}{2}\log N$; with 3-level variables each extra parent triples a CPT, so it often stops with a sparse graph.</details>
8. In which entry of the method map would you put each of the three classifiers of this capstone, and which of the course's three ideas does each illustrate?
   <details><summary>Answer</summary>L1-logistic: discriminative, probabilistic — the two-step template (Bernoulli model + Laplace prior → MAP). Random forest: non-probabilistic (classification) — variance reduction by averaging; its cousins are probabilistic ensembles. RBF SVM: non-probabilistic — its probabilistic cousin is logistic regression (hinge vs logistic loss).</details>

## 🧠 Questions & discussion

1. Our three classifiers were statistically indistinguishable. What kind of data set (size, signal, structure) would you need to *rank* them — and how would you estimate the required number of patients before collecting data?
2. The development and test sets came from different collections. Give two ways a distribution shift could make a classifier fail on the test set even though its nested-CV estimate was excellent.
3. Is accuracy the right metric for ALL vs AML? Think about the consequences of each type of error and about class imbalance; propose an alternative metric and how it would change the contract.
4. A colleague wants to publish the 10 most frequently selected genes as 'the AML signature'. What analyses and experiments would you require first?
5. For the Sachs mini-project: the consensus network itself was assembled from the literature. How would you evaluate network learners if no trustworthy 'truth' existed?
6. Which of the three recurring ideas of the course did you find most useful outside this course, and why?

## 🚀 Bonus — going further

- **A second data set.** Run Sections 2–8 unchanged on the Wisconsin breast-cancer data (`sklearn.datasets.load_breast_cancer`, 569 tumours × 30 features; set `K_GENES = 30`). With 15 times more samples, can the corrected paired test now separate the methods?
- **Replace a method.** Swap the random forest for Gaussian naïve Bayes or k-NN — the extra case-study slides of `015` compare kNN, a Gaussian classifier and logistic regression on Golub — and redo Sections 5–8.
- **Uncertainty you can hand to a clinician.** Conformal prediction turns any classifier into prediction *sets* with guaranteed coverage: see [`ConformalPrediction`](../../ConformalPrediction/) in this repository.
- **Deeper dives in this repository.** Random forests on real TCGA driver mutations ([`DecisionTreesRandomForests`](../../DecisionTreesRandomForests/)); regularised regression on wheat genomics ([`RegularizedLinearRegression`](../../RegularizedLinearRegression/)); generative vs discriminative classifiers ([`GenerativeDiscriminativeClassification`](../../GenerativeDiscriminativeClassification/)); MLE/MAP and uncertainty ([`FundamentalsLearningUncertainty`](../../FundamentalsLearningUncertainty/)).
- **The sister capstone.** [`BiologyToResearch`](../../../../cryoem/tutorial/BiologyToResearch/) ends with an end-to-end visual-proteomics study on a sealed in-silico cell and a paper audit — the same discipline (sealed data, honest evaluation, calibrated claims) applied to cryo-electron tomography.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 24 — Capstone project
Input: Golub leukaemia data (38 development + 34 sealed test patients × 3,571 genes); Sachs cells (1,800 discrete / 2,603 continuous × 11 proteins)
Output: evaluation contract; nested-CV accuracies; paired comparisons; test accuracies with CIs; stable genes; three learned networks with precision/recall
Method: in-fold t-filter → {L1-logistic, random forest, RBF SVM}; nested CV (5×2 outer, 3 inner); single test evaluation;
        Chow–Liu / BIC hill climbing / graphical lasso vs the consensus skeleton
Key equation: SE_corr = sqrt((1/F + n_test/n_train) s_d²);  McNemar b ~ Binomial(b + c, 1/2);
              BIC score = log L − (k/2) log N  (= −½ × Module 23's BIC, so here higher is better)
Key parameters: K_GENES = 200; tuning grids (4 settings per method); n_outer, n_repeats, n_inner; graphical-lasso α
What changed when I modified … (n_repeats, K_GENES, a method, the data set, α):
One failure mode: (e.g. choosing genes outside the folds; declaring a winner from overlapping CIs; reading arrows as causation)
Connection to 02-620 slides: 001_intro 9–10 (project); 015 30–37 (CV); 008 157–172 and 009 49–58 (structure learning); all decks (Section 11)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, decks `001_intro`, `002_regression`, `004_classification-LR`, `005_decision-tree`, `006_clustering`, `007_SVM`, `008_BN_mz_021825`, `009_GGM_mz_030925`, `011_PCA_mz_031825`, `015_model-selection_mz_040725` and `015_model-selection--extra` (course design S. Kim; instructors M. Xu and M. Zhang); Section 11 summarises all decks.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §7.10 (cross-validation, including 'the wrong and right way'), Chapter 15 (random forests), Chapter 12 (SVMs), §17.3 (graphical lasso).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapters 6 (frequentist statistics, model selection), 13 (sparse linear models), 26 (graphical-model structure learning).
- Golub T. R., Slonim D. K., Tamayo P., et al. (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *JASA* 97:77–87.
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *PNAS* 99:6562–6566.
- Nadeau C., Bengio Y. (2003) Inference for the generalization error. *Machine Learning* 52:239–281. · Dietterich T. G. (1998) Approximate statistical tests for comparing supervised classification learning algorithms. *Neural Computation* 10:1895–1923.
- Tibshirani R. (1996) Regression shrinkage and selection via the lasso. *JRSS-B* 58:267–288. · Breiman L. (2001) Random forests. *Machine Learning* 45:5–32. · Cortes C., Vapnik V. (1995) Support-vector networks. *Machine Learning* 20:273–297.
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from multiparameter single-cell data. *Science* 308:523–529.
- Chow C. K., Liu C. N. (1968) Approximating discrete probability distributions with dependence trees. *IEEE Trans Inf Theory* 14:462–467. · Schwarz G. (1978) Estimating the dimension of a model. *Ann Stat* 6:461–464. · Friedman J., Hastie T., Tibshirani R. (2008) Sparse inverse covariance estimation with the graphical lasso. *Biostatistics* 9:432–441.

## ➡️ Next

This is the last module of BiologyToML — congratulations on finishing the course. Next: run your own project with the
templates above, and revisit any module whose self-assessment box is still empty, starting from
[Module 00](00_welcome_ml_for_biology.ipynb)'s roadmap. For more, explore the deep-dive tutorials linked in the 🚀 Bonus.